In [1]:

from typing import Dict, Tuple
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import models, transforms
from torchvision.datasets import MNIST
from torchvision.utils import save_image, make_grid
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
import numpy as np

%matplotlib inline

from torch import autograd
from torch.autograd import Variable
from tensorboardX import SummaryWriter
import torch.optim as optim
import torchvision.datasets as datasets
import time
import os

if __name__ == "__main__":
    print("Torch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("CUDA version:", torch.version.cuda)
    print("Number of GPUs:", torch.cuda.device_count())
    print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.device_count() > 0 else "No GPU detected")

Torch version: 2.7.0+cu126
CUDA available: True
CUDA version: 12.6
Number of GPUs: 1
GPU name: NVIDIA GeForce RTX 4090


In [ ]:
class CombinedModeWeightNet(nn.Module):
    def __init__(self, mode_model, weight_model):
        super().__init__()
        self.mode_model = mode_model
        self.weight_model = weight_model

    def forward(self, x_img, x_cond):
        mode = self.mode_model(x_img, x_cond)   # [B, 1]
        weight = self.weight_model(x_img, x_cond)  # [B, 1]
        return torch.cat((mode, weight), dim=1)   # [B, 2]

from only_mode_only_weight_v3 import No_normal_modewieght_net

device = 'cuda' if torch.cuda.is_available() else 'cpu'

mode0_model = No_normal_modewieght_net().to(device)
weight0_model = No_normal_modewieght_net().to(device)

mode0_model.load_state_dict(torch.load('models/only_first_mode_no_normalization_with_less_dropout.pth', map_location=torch.device(device)))
weight0_model.load_state_dict(torch.load('models/only_first_weight_no_normalization_with_less_dropout_100.pth', map_location=torch.device(device)))

cnn = CombinedModeWeightNet(mode0_model, weight0_model).to(device)
cnn.eval()

In [2]:
def plot_shape(shape_matrix):
    """Plot the generated shape (expects input shape (1, 32, 32) or (32, 32))."""
    # Squeeze channel if present
    if shape_matrix.ndim == 3 and shape_matrix.shape[0] == 1:
        shape_matrix = shape_matrix.squeeze(0)  # → (32, 32)

    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(6, 6))
    ax.set_facecolor('#301934')
    ax.imshow(shape_matrix, origin='upper', cmap='viridis')  # add colormap if needed
    plt.axis('off')
    # print(f'size: {shape_matrix.shape[0]} x {shape_matrix.shape[1]}')
    plt.show()


def load_item(item, p= True, action=''):
    if action=='':
        if p:
            print(f'Cond: {item[0]}')
            print(f'Params: {item[1]}')
        plot_shape(item[2])
        return {'Cond':item[0], 'Params':item[1]}
    if action == 'shape':
        return item[3]
    
def quarter(matrix):
    return matrix[:32, :32]

In [3]:
class ResidualConvBlock(nn.Module):
    def __init__(
        self, in_channels: int, out_channels: int, is_res: bool = False
    ) -> None:
        super().__init__()
        '''
        standard ResNet style convolutional block, for image processing
        '''
        self.same_channels = in_channels == out_channels
        self.is_res = is_res
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, 1, 1),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 3, 1, 1),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.is_res:
            x1 = self.conv1(x)
            x2 = self.conv2(x1)
            # this adds on correct residual in case channels have increased
            if self.same_channels:
                out = x + x2
            else:
                out = x1 + x2
            return out / 1.414
        else:
            x1 = self.conv1(x)
            x2 = self.conv2(x1)
            return x2


In [4]:
class UnetDown(nn.Module):
    """
    Downsampling path for U-Net, reduces spatial resolution while increasing feature depth
    Input: Image batch, size (batchsize, 1, 32, 32)
    Output: size (batchsize, out_channels, 16, 16)
    Output:
    """
    def __init__(self, in_channels, out_channels):
        super(UnetDown, self).__init__()
        '''
        process and downscale the image feature maps
        '''
        layers = [ResidualConvBlock(
            in_channels, out_channels), nn.MaxPool2d(2)]
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        # Doubles spatial dimensions, halves feature dimensions
        # My channel dimension for image will always be 1, greyscale
        return self.model(x)


In [5]:
class UnetUp(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UnetUp, self).__init__()
        '''
        process and upscale the image feature maps
        Doubles spatial size, but decreases channels:
        input: 2 vectors of size (binsize, in_channels / 2, h, w) 
        output: (binsize, outchannels, 2h, 2w)
        '''
        layers = [
            nn.ConvTranspose2d(in_channels, out_channels, 2, 2),
            ResidualConvBlock(out_channels, out_channels),
            ResidualConvBlock(out_channels, out_channels),
        ]
        self.model = nn.Sequential(*layers)

    def forward(self, x, skip):
        """
        x is the upsampled features from previous decoder layer
        skip is the skip connection from the encoder, same size as x
        """
        x = torch.cat((x, skip), 1)
        x = self.model(x)
        return x

In [6]:
class EmbedFC(nn.Module):
    """
    Use FC layer for embedding 1-d metadata, like modes+weights
    (putting into higher dimension)
    Effectively our conditional
    input: Conditional, size (batchsize, input_dim = 4+4)
    Output: Higherdimensional tensor, size (batchsize, output_dim)
    
    """
    def __init__(self, input_dim, emb_dim):
        super(EmbedFC, self).__init__()
        '''
        generic one layer FC NN for embedding things  
        '''
        self.input_dim = input_dim
        layers = [
            nn.Linear(input_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, emb_dim),
        ]
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        x = x.view(-1, self.input_dim)
        return self.model(x)

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Assumes you already have these:
# - ResidualConvBlock(in_ch, out_ch, is_res=True)
# - UnetDown(in_ch, out_ch)
# - UnetUp(in_ch, out_ch)
# - EmbedFC(in_dim, out_dim)

class ContextUnet(nn.Module):
    """
    U-Net for conditional image generation (32x32), where conditioning is
    provided by *tiled* scalar maps concatenated to the input channels.

    Conditioning (this version):
      - top (mode, weight): shape [B, 2]  (or [B, 1, 2], both supported)
      - params:             shape [B, 4]
      -> total 6 scalars per sample, each tiled to [H, W] and concatenated as channels.

    Timestep t is still embedded via an MLP and injected on the up path.
    """

    def __init__(self, in_channels=1, n_feat=256, use_time_embed=True):
        super().__init__()
        self.in_channels = in_channels
        self.n_feat = n_feat
        self.use_time_embed = use_time_embed

        # 6 tiled condition channels (2 from top mode/weight + 4 params)
        self.cond_channels = 6
        lifted_in = in_channels + self.cond_channels

        # Encoder
        self.init_conv = ResidualConvBlock(lifted_in, n_feat, is_res=True)
        self.down1 = UnetDown(n_feat, n_feat)          # 32x32 -> 16x16 (n_feat)
        self.down2 = UnetDown(n_feat, 2 * n_feat)      # 16x16 -> 8x8   (2*n_feat)

        # Latent pooling to 1x1
        self.to_vec = nn.Sequential(nn.AvgPool2d(8), nn.GELU())

        # Optional timestep embedding (kept as MLP, not tiled)
        if use_time_embed:
            self.timeembed1 = EmbedFC(1, 2 * n_feat)
            self.timeembed2 = EmbedFC(1, 1 * n_feat)

        # Decoder
        self.up0 = nn.Sequential(
            nn.ConvTranspose2d(2 * n_feat, 2 * n_feat, 8, 8),  # 1x1 -> 8x8
            nn.GroupNorm(8, 2 * n_feat),
            nn.ReLU(),
        )
        # Concats with skips: (2*n_feat up) + (2*n_feat skip) = 4*n_feat -> n_feat
        self.up1 = UnetUp(4 * n_feat, n_feat)          # 8x8 -> 16x16
        # (n_feat up) + (n_feat skip) = 2*n_feat -> n_feat
        self.up2 = UnetUp(2 * n_feat, n_feat)          # 16x16 -> 32x32

        # Output head: concat with early skip (x after init_conv) -> 2*n_feat
        self.out = nn.Sequential(
            nn.Conv2d(2 * n_feat, n_feat, 3, 1, 1),
            nn.GroupNorm(8, n_feat),
            nn.ReLU(),
            nn.Conv2d(n_feat, in_channels, 3, 1, 1),
        )

    @staticmethod
    def _tile_condition(top_pair, params, H, W):
        """
        top_pair: [B, 2] or [B, 1, 2]  (mode_0, weight_0)
        params:   [B, 4]
        Returns tiled tensor of shape [B, 6, H, W]
        """
        if top_pair.dim() == 3:
            # allow [B, 1, 2] → [B, 2]
            assert top_pair.size(1) == 1 and top_pair.size(2) == 2, \
                "top_pair must be [B, 2] or [B, 1, 2]"
            top_pair = top_pair.squeeze(1)
        else:
            assert top_pair.dim() == 2 and top_pair.size(1) == 2, \
                "top_pair must be [B, 2] (mode, weight)"

        B = top_pair.shape[0]
        cond = torch.cat([top_pair, params], dim=1)    # [B, 6]
        cond = cond.unsqueeze(-1).unsqueeze(-1)        # [B, 6, 1, 1]
        cond = cond.repeat(1, 1, H, W)                 # [B, 6, H, W]
        return cond

    def forward(self, x, top_pair, params, t):
        """
        x:        [B, 1, 32, 32]  (noisy waveguide / latent)
        top_pair: [B, 2] or [B, 1, 2]  (mode_0, weight_0)
        params:   [B, 4]
        t:        [B, 1] (scalar timestep per sample)
        """
        B, _, H, W = x.shape
        assert H == 32 and W == 32, "This U-Net assumes 32x32 spatial size."

        # Build and add tiled conditions as channels at the input
        cond_maps = self._tile_condition(top_pair, params, H, W)  # [B, 6, 32, 32]
        x_in = torch.cat([x, cond_maps], dim=1)                   # [B, 1+6, 32, 32]

        # Encoder
        x0 = self.init_conv(x_in)   # [B, n_feat, 32, 32]
        d1 = self.down1(x0)         # [B, n_feat, 16, 16]
        d2 = self.down2(d1)         # [B, 2*n_feat, 8, 8]
        h  = self.to_vec(d2)        # [B, 2*n_feat, 1, 1]

        # Decode (optionally inject time embeddings)
        up1 = self.up0(h)           # [B, 2*n_feat, 8, 8]

        if self.use_time_embed:
            temb1 = self.timeembed1(t).view(B, 2 * self.n_feat, 1, 1)
            temb2 = self.timeembed2(t).view(B, 1 * self.n_feat, 1, 1)
            up1 = up1 + temb1

        u2 = self.up1(up1, d2)      # -> [B, n_feat, 16, 16]
        if self.use_time_embed:
            u2 = u2 + temb2

        u3 = self.up2(u2, d1)       # -> [B, n_feat, 32, 32]

        # Final head with early skip (x0)
        out = self.out(torch.cat([u3, x0], dim=1))  # [B, 1, 32, 32]
        return out


In [8]:
def ddpm_schedules(beta1, beta2, T):
    """
    Precomputes all noise scheduling terms needed for training and sampling
    from a denoising diffusion probabilistic model
    Uses a sequence of gradually increasing noise level over T timesteps
    beta1: starting noise level, O(1e-4)
    beta2: final noise level, O(0.02)
    T: number of time steps
    """
    assert beta1 < beta2 < 1.0, "beta1 and beta2 must be in (0, 1)"

    beta_t = (beta2 - beta1) * torch.arange(0, T + 1, dtype=torch.float32) / T + beta1 # noise variance schedule (for every time t in T)
    sqrt_beta_t = torch.sqrt(beta_t)
    alpha_t = 1 - beta_t
    log_alpha_t = torch.log(alpha_t)
    alphabar_t = torch.cumsum(log_alpha_t, dim=0).exp()

    sqrtab = torch.sqrt(alphabar_t)
    oneover_sqrta = 1 / torch.sqrt(alpha_t)

    sqrtmab = torch.sqrt(1 - alphabar_t)
    mab_over_sqrtmab_inv = (1 - alpha_t) / sqrtmab

    # dictionary of schedule terms
    return {
        "alpha_t": alpha_t,  # \alpha_t , signal retention at time step t
        "oneover_sqrta": oneover_sqrta,  # 1/\sqrt{\alpha_t}
        "sqrt_beta_t": sqrt_beta_t,  # \sqrt{\beta_t} , noise scaling factor
        "alphabar_t": alphabar_t,  # \bar{\alpha_t} , cumulative signal retention
        "sqrtab": sqrtab,  # \sqrt{\bar{\alpha_t}} , scales clean image during noise
        "sqrtmab": sqrtmab,  # \sqrt{1-\bar{\alpha_t}} , noise strength
        "mab_over_sqrtmab": mab_over_sqrtmab_inv,  # (1-\alpha_t)/\sqrt{1-\bar{\alpha_t}} , for reverse diffusion
    }


In [9]:
import torch
import torch.nn as nn
import numpy as np

class DDPM(nn.Module):
    def __init__(self, nn_model, betas, n_T, device, drop_prob=0.1):
        super().__init__()
        self.nn_model = nn_model.to(device)

        # register all schedule buffers
        sched = ddpm_schedules(betas[0], betas[1], n_T)
        for k, v in sched.items():
            self.register_buffer(k, v)

        self.n_T = n_T
        self.device = device
        self.drop_prob = drop_prob
        self.loss_mse = nn.MSELoss()

    @staticmethod
    def _apply_mask(top_pair, params, context_mask):
        """
        context_mask: [B] or [B,1] of {0,1}, where 1 => drop conditioning.
        Returns masked copies (zeros when dropped).

        top_pair: [B, 2] or [B, 1, 2]
        params:   [B, 4]
        """
        if context_mask.dim() == 1:
            context_mask = context_mask.unsqueeze(1)  # [B,1]

        # Normalize top_pair shape to [B, 2]
        if top_pair.dim() == 3:
            assert top_pair.size(1) == 1 and top_pair.size(2) == 2, \
                "top_pair must be [B, 2] or [B, 1, 2]"
            top_pair = top_pair.squeeze(1)
        else:
            assert top_pair.dim() == 2 and top_pair.size(1) == 2, \
                "top_pair must be [B, 2]"

        tp = top_pair * (1.0 - context_mask)            # [B,2] broadcast
        pr = params   * (1.0 - context_mask)            # [B,4] broadcast
        return tp, pr

    def forward(self, x, top_pair, params):
        """
        Training step:
          x:        [B,1,32,32] (clean image x0)
          top_pair: [B,2] (mode_0, weight_0)  or [B,1,2]
          params:   [B,4]
        """
        B = x.shape[0]
        _ts = torch.randint(1, self.n_T + 1, (B,), device=self.device)        # [B]
        noise = torch.randn_like(x)                                           # [B,1,32,32]

        x_t = (
            self.sqrtab[_ts, None, None, None] * x
            + self.sqrtmab[_ts, None, None, None] * noise
        )

        # classifier-free dropout
        context_mask = torch.bernoulli(
            torch.full((B,), self.drop_prob, device=self.device)
        )  # [B] in {0,1}

        tp_masked, pr_masked = self._apply_mask(top_pair, params, context_mask)

        # normalized timestep as [B,1]
        t_norm = (_ts.float() / self.n_T).unsqueeze(1)  # [B,1]

        pred_noise = self.nn_model(x_t, tp_masked, pr_masked, t_norm)
        return self.loss_mse(noise, pred_noise)

    @torch.no_grad()
    def sample(self, n_sample, size, device, top_pair, params, guide_w=0.0):
        """
        Sampling with classifier-free guidance (CFG).

        Args
        ----
        n_sample: int
        size:     tuple like (1, 32, 32)
        device:   torch.device
        top_pair: [n_sample, 2] tensor (mode_0, weight_0)  or [n_sample, 1, 2]
        params:   [n_sample, 4] tensor
        guide_w:  float guidance scale (0 = no CFG)

        Returns
        -------
        x_T->x_0 sample tensor [n_sample, 1, 32, 32], and numpy trajectory.
        """
        assert top_pair.shape[0] == n_sample and params.shape[0] == n_sample

        x_i = torch.randn(n_sample, *size, device=device)

        # Build masks for double batch (first half conditioned, second half dropped)
        context_mask_cond   = torch.zeros(n_sample, device=device)  # keep
        context_mask_uncond = torch.ones(n_sample,  device=device)  # drop

        # Precompute cond/uncond views
        tp_cond, pr_cond       = self._apply_mask(top_pair, params, context_mask_cond)
        tp_uncond, pr_uncond   = self._apply_mask(top_pair, params, context_mask_uncond)

        x_i_store = []
        for i in range(self.n_T, 0, -1):
            # timestep scalar normalized
            t_norm = torch.full((n_sample, 1), i / self.n_T, device=device)

            # double the batch (conditioned + unconditioned)
            x_in = torch.cat([x_i, x_i], dim=0)
            t_in = torch.cat([t_norm, t_norm], dim=0)             # [2B,1]
            tp_in = torch.cat([tp_cond, tp_uncond], dim=0)        # [2B,2]
            pr_in = torch.cat([pr_cond, pr_uncond], dim=0)        # [2B,4]

            # predict noise for both halves
            eps = self.nn_model(x_in, tp_in, pr_in, t_in)         # [2B,1,32,32]
            eps1, eps2 = eps[:n_sample], eps[n_sample:]           # cond, uncond

            # CFG combine
            eps_cfg = (1 + guide_w) * eps1 - guide_w * eps2

            z = torch.randn_like(x_i) if i > 1 else 0.0
            x_i = (
                self.oneover_sqrta[i] * (x_i - eps_cfg * self.mab_over_sqrtmab[i])
                + self.sqrt_beta_t[i] * z
            )

            if i % 20 == 0 or i == self.n_T or i < 8:
                x_i_store.append(x_i.detach().cpu().numpy())

        x_i_store = np.array(x_i_store)
        return x_i, x_i_store


In [10]:
import importlib
import waveguide_dataset_paired_wtest
importlib.reload(waveguide_dataset_paired_wtest)
from waveguide_dataset_paired_wtest import WaveguideDatasetPaired

In [28]:
import os
import torch
import torch.nn.functional as F
import numpy as np
from torch.utils.data import DataLoader
from torchvision.utils import make_grid, save_image
from tqdm import tqdm
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

def train_waveguide_ddpm(
    h5_path='train_test_split.h5',
    stats_path="waveguide_stats_log_norm_above90.npz",
    save_dir="./data/diffusion_params_v4_topmode_CNN_eval/",
    n_epoch=20,
    batch_size=256,
    n_T=400,
    n_feat=128,
    lrate=1e-4,
    drop_prob=0.1,
    betas=(1e-4, 0.02),
    ws_test=(0.0, 0.5, 2.0),
    save_model=True,
    test_eval_fraction=0.25,   # portion of test loader to evaluate each epoch
    device=None,
    cnn=None,                  # <-- NEW: frozen evaluator CNN (unnormalized I/O)
    ddpm_state=None,
):
    assert cnn is not None, "Pass your preloaded CNN as cnn=..."
    os.makedirs(save_dir, exist_ok=True)
    device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")

    # ---- Data ----
    train_ds = WaveguideDatasetPaired(h5_path, split="train", stats_path=stats_path)
    test_ds  = WaveguideDatasetPaired(h5_path, split="test",  stats_path=stats_path)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)

    # ---- Model ----
    # ContextUnet expects (x, top_pair, params, t)
    unet = ContextUnet(in_channels=1, n_feat=n_feat, use_time_embed=True)
    ddpm = DDPM(nn_model=unet, betas=betas, n_T=n_T, device=device, drop_prob=drop_prob).to(device)

    if ddpm_state != None:
        ddpm.load_state_dict(torch.load(ddpm_state, map_location=torch.device(device)))

    optim = torch.optim.Adam(ddpm.parameters(), lr=lrate)

    # Freeze CNN, allow grads to flow to its inputs (generated x), not its weights.
    cnn = cnn.to(device).eval()
    for p in cnn.parameters():
        p.requires_grad_(False)

    # ---- Helpers ----
    def extract_top_pair(cond_b42):
        """
        cond_b42: [B, 4, 2]  (modes, weights) in order of importance.
        Return top (mode, weight) as [B, 2].
        """
        assert cond_b42.dim() == 3 and cond_b42.size(1) >= 1 and cond_b42.size(2) == 2, \
            f"Expected [B,4,2]-like, got {tuple(cond_b42.shape)}"
        return cond_b42[:, 0, :]  # [B,2] → (top mode, top weight)

    # Get param denorm stats (prefer from dataset; fall back to stats file)
    def get_param_stats():
        # Try attributes on dataset (common pattern)
        meanp = getattr(train_ds, "meanp", None)
        stdp  = getattr(train_ds, "stdp", None)
        if meanp is not None and stdp is not None:
            meanp = torch.as_tensor(meanp, dtype=torch.float32, device=device)
            stdp  = torch.as_tensor(stdp,  dtype=torch.float32, device=device)
            return meanp, stdp
        # Fallback to npz
        stats = np.load(stats_path)
        meanp = torch.tensor(stats["meanp"], dtype=torch.float32, device=device)
        stdp  = torch.tensor(stats["stdp"],  dtype=torch.float32, device=device)
        return meanp, stdp

    param_mean, param_std = get_param_stats()

    def denorm_params(params_norm):
        # params_norm: [B,4] (normalized in dataset)
        return params_norm * param_std + param_mean

    # ---- Tracking ----
    train_losses, test_losses = [], []
    best_test_loss = float("inf")

    @torch.no_grad()
    def evaluate_on_test_portion():
        """CNN-perceptual loss on a portion of test set (no CFG dropout in U-Net)."""
        ddpm.eval()
        was = ddpm.drop_prob
        ddpm.drop_prob = 0.0  # disable classifier-free dropout during eval

        total, count = 0.0, 0
        max_batches = max(1, int(np.ceil(test_eval_fraction * len(test_loader)))) if len(test_loader) > 0 else 1

        for b_idx, (cond, params, x_real) in enumerate(test_loader):
            x_real = x_real.to(device, non_blocking=True)  # [B,1,32,32]
            cond   = cond.to(device, non_blocking=True)    # [B,4,2] (normalized)
            params = params.to(device, non_blocking=True)  # [B,4]   (normalized)
            top_pair = extract_top_pair(cond)              # [B,2]

            # Random t and noise like in training
            B = x_real.size(0)
            _ts   = torch.randint(1, n_T + 1, (B,), device=device)
            noise = torch.randn_like(x_real)
            x_t   = ddpm.sqrtab[_ts, None, None, None] * x_real + ddpm.sqrtmab[_ts, None, None, None] * noise
            t_norm = (_ts.float() / n_T).unsqueeze(1)

            # No dropout at eval
            pred_noise = ddpm.nn_model(x_t, top_pair, params, t_norm)
            x0_hat = (x_t - ddpm.sqrtmab[_ts, None, None, None] * pred_noise) / ddpm.sqrtab[_ts, None, None, None]
            # Optionally clamp to valid range if your CNN expects [0,1]
            x0_hat = x0_hat.clamp(0.0, 1.0)

            # Denormalize params for CNN
            params_un = denorm_params(params)

            # CNN outputs (unnormalized top mode/weight)
            pred_top = cnn(x0_hat, params_un)     # [B,2]
            targ_top = cnn(x_real,  params_un)    # [B,2]

            loss = F.mse_loss(pred_top, targ_top)
            total += float(loss.item())
            count += 1
            if b_idx + 1 >= max_batches:
                break

        ddpm.drop_prob = was
        return total / max(count, 1)

    # ---- Training loop ----
    for ep in range(n_epoch):
        print(f"epoch {ep}")
        ddpm.train()

        # linear LR decay
        for g in optim.param_groups:
            g["lr"] = lrate * (1 - ep / n_epoch)

        pbar = tqdm(train_loader)
        loss_ema = None
        train_loss_sum, train_loss_batches = 0.0, 0

        for cond, params, x_real in pbar:
            x_real = x_real.to(device, non_blocking=True)  # [B,1,32,32] (normalized image)
            cond   = cond.to(device, non_blocking=True)    # [B,4,2]     (normalized)
            params = params.to(device, non_blocking=True)  # [B,4]       (normalized)

            top_pair = extract_top_pair(cond)              # [B,2]

            # ---- Draw a random step and construct x_t ----
            B = x_real.size(0)
            _ts   = torch.randint(1, n_T + 1, (B,), device=device)     # [B]
            noise = torch.randn_like(x_real)                            # [B,1,32,32]
            x_t   = ddpm.sqrtab[_ts, None, None, None] * x_real + ddpm.sqrtmab[_ts, None, None, None] * noise
            t_norm = (_ts.float() / n_T).unsqueeze(1)                   # [B,1]

            # ---- Classifier-free dropout on conditioning ----
            context_mask = torch.bernoulli(torch.full((B,), ddpm.drop_prob, device=device))  # [B] in {0,1}
            ctx = context_mask.unsqueeze(1)  # [B,1]

            # mask top_pair [B,2] and params [B,4] when dropped
            top_pair_masked = top_pair * (1.0 - ctx)
            params_masked   = params   * (1.0 - ctx)

            # ---- Predict noise and reconstruct x0 ----
            optim.zero_grad()
            pred_noise = ddpm.nn_model(x_t, top_pair_masked, params_masked, t_norm)  # [B,1,32,32]
            x0_hat = (x_t - ddpm.sqrtmab[_ts, None, None, None] * pred_noise) / ddpm.sqrtab[_ts, None, None, None]
            # If your CNN expects [0,1] inputs, uncomment the clamp:
            # x0_hat = x0_hat.clamp(0.0, 1.0)

            # ---- Denormalize params (ONLY) for CNN ----
            params_un = denorm_params(params)

            # ---- CNN perceptual targets and predictions ----
            # Target uses the REAL waveguide; no grad needed
            with torch.no_grad():
                targ_top = cnn(x_real, params_un)   # [B,2], unnormalized

            # Prediction uses the GENERATED waveguide; grad MUST flow back to ddpm
            pred_top = cnn(x0_hat, params_un)       # [B,2], unnormalized

            # ---- Perceptual loss in (mode, weight) space ----
            loss = F.mse_loss(pred_top, targ_top)
            loss.backward()
            optim.step()

            # ---- Logging ----
            loss_val = float(loss.detach().item())
            train_loss_sum += loss_val
            train_loss_batches += 1
            loss_ema = loss_val if loss_ema is None else (0.95 * loss_ema + 0.05 * loss_val)
            pbar.set_description(f"cnn-loss: {loss_ema:.5f}")

        # Mean train loss this epoch
        mean_train_loss = train_loss_sum / max(train_loss_batches, 1)
        train_losses.append(mean_train_loss)

        # ---- Evaluation on test subset (CNN metric) ----
        mean_test_loss = evaluate_on_test_portion()
        test_losses.append(mean_test_loss)
        print(f"epoch {ep}: train(cnn)={mean_train_loss:.6f} | test(cnn)={mean_test_loss:.6f}")

        # ---- Save best model ----
        if save_model and mean_test_loss < best_test_loss:
            best_test_loss = mean_test_loss
            best_path = os.path.join(save_dir, "best_model.pth")
            torch.save(ddpm.state_dict(), best_path)
            print(f"✔ improved test loss; saved best model to {best_path}")

        # ---- Visualization (samples & GIF) ----
        ddpm.eval()
        with torch.no_grad():
            try:
                cond_eval, params_eval, x_real_batch = next(iter(test_loader))
            except StopIteration:
                cond_eval, params_eval, x_real_batch = next(
                    iter(DataLoader(test_ds, batch_size=min(32, len(test_ds))))
                )

            cond_eval    = cond_eval.to(device)    # [B,4,2]
            params_eval  = params_eval.to(device)  # [B,4]
            x_real_batch = x_real_batch.to(device) # [B,1,32,32]

            n_sample = min(32, cond_eval.shape[0])
            cond_eval    = cond_eval[:n_sample]
            params_eval  = params_eval[:n_sample]
            x_real_vis   = x_real_batch[:n_sample]
            top_pair_eval = extract_top_pair(cond_eval)  # [B,2]

            for w in ws_test:
                x_gen, x_gen_store = ddpm.sample(
                    n_sample=n_sample,
                    size=(1, 32, 32),
                    device=device,
                    top_pair=top_pair_eval,
                    params=params_eval,
                    guide_w=w,
                )

                x_all = torch.cat([x_gen, x_real_vis], dim=0)
                grid = make_grid(x_all, nrow=int(np.sqrt(2*n_sample)))
                save_path = os.path.join(save_dir, f"image_ep{ep}_w{w}.png")
                save_image(grid, save_path)
                print(f"saved image at {save_path}")

                if ep % 5 == 0 or ep == int(n_epoch - 1):
                    import numpy as np
                    fig, axs = plt.subplots(
                        nrows=int(n_sample // 8) if n_sample >= 8 else 1,
                        ncols=min(8, n_sample),
                        sharex=True, sharey=True, figsize=(8, 3),
                    )
                    axs = np.atleast_2d(axs)

                    def animate_diff(i, store):
                        plots = []
                        frame = -store[i]
                        vmin, vmax = frame.min(), frame.max()
                        idx = 0
                        for r in range(axs.shape[0]):
                            for c in range(axs.shape[1]):
                                if idx >= n_sample: break
                                axs[r, c].clear()
                                axs[r, c].set_xticks([]); axs[r, c].set_yticks([])
                                plots.append(axs[r, c].imshow(frame[idx, 0], cmap="gray", vmin=vmin, vmax=vmax))
                                idx += 1
                        return plots

                    ani = FuncAnimation(fig, animate_diff, fargs=[x_gen_store], interval=200,
                                        blit=False, repeat=True, frames=x_gen_store.shape[0])
                    gif_path = os.path.join(save_dir, f"gif_ep{ep}_w{w}.gif")
                    ani.save(gif_path, dpi=100, writer=PillowWriter(fps=5))
                    plt.close(fig)
                    print(f"saved gif at {gif_path}")

        # ---- Plot & save loss curves (updates every epoch) ----
        try:
            plt.figure(figsize=(6,4))
            plt.plot(range(1, len(train_losses)+1), train_losses, label="Train CNN-MSE")
            plt.plot(range(1, len(test_losses)+1),  test_losses,  label="Test CNN-MSE (subset)")
            plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Train vs Test (CNN space)")
            plt.legend(); plt.tight_layout()
            loss_curve_path = os.path.join(save_dir, "loss_curve.png")
            plt.savefig(loss_curve_path, dpi=150)
            plt.close()
            print(f"updated loss curve at {loss_curve_path}")
        except Exception as e:
            print(f"warning: failed to plot loss curve ({e})")

    # Optionally save final model state
    if save_model:
        final_path = os.path.join(save_dir, f"model_final.pth")
        torch.save(ddpm.state_dict(), final_path)
        print(f"saved final model at {final_path}")


In [29]:
if __name__ == "__main__":
    train_waveguide_ddpm(cnn=cnn, ddpm_state='data/diffusion_params_v3_topmode/best_model.pth')

epoch 0


loss: 0.0107: 100%|██████████| 3501/3501 [07:45<00:00,  7.53it/s]


epoch 0: train_loss=0.029532 | test_loss=0.011939
✔ improved test loss; saved best model to ./data/diffusion_params_v1_deep/best_model.pth
saved image at ./data/diffusion_params_v1_deep/image_ep0_w0.0.png
saved gif at ./data/diffusion_params_v1_deep/gif_ep0_w0.0.gif
saved image at ./data/diffusion_params_v1_deep/image_ep0_w0.5.png
saved gif at ./data/diffusion_params_v1_deep/gif_ep0_w0.5.gif
saved image at ./data/diffusion_params_v1_deep/image_ep0_w2.0.png
saved gif at ./data/diffusion_params_v1_deep/gif_ep0_w2.0.gif
updated loss curve at ./data/diffusion_params_v1_deep/loss_curve.png
epoch 1


loss: 0.0072: 100%|██████████| 3501/3501 [07:46<00:00,  7.50it/s]


epoch 1: train_loss=0.008525 | test_loss=0.007466
✔ improved test loss; saved best model to ./data/diffusion_params_v1_deep/best_model.pth
saved image at ./data/diffusion_params_v1_deep/image_ep1_w0.0.png
saved image at ./data/diffusion_params_v1_deep/image_ep1_w0.5.png
saved image at ./data/diffusion_params_v1_deep/image_ep1_w2.0.png
updated loss curve at ./data/diffusion_params_v1_deep/loss_curve.png
epoch 2


...

epoch 17: train_loss=0.003705 | test_loss=0.003798
✔ improved test loss; saved best model to ./data/diffusion_params_v1_deep/best_model.pth
saved image at ./data/diffusion_params_v1_deep/image_ep17_w0.0.png
saved image at ./data/diffusion_params_v1_deep/image_ep17_w0.5.png
saved image at ./data/diffusion_params_v1_deep/image_ep17_w2.0.png
updated loss curve at ./data/diffusion_params_v1_deep/loss_curve.png
epoch 18


loss: 0.0037: 100%|██████████| 3501/3501 [07:14<00:00,  8.05it/s]


epoch 18: train_loss=0.003656 | test_loss=0.003746
✔ improved test loss; saved best model to ./data/diffusion_params_v1_deep/best_model.pth
saved image at ./data/diffusion_params_v1_deep/image_ep18_w0.0.png
saved image at ./data/diffusion_params_v1_deep/image_ep18_w0.5.png
saved image at ./data/diffusion_params_v1_deep/image_ep18_w2.0.png
updated loss curve at ./data/diffusion_params_v1_deep/loss_curve.png
epoch 19


loss: 0.0036: 100%|██████████| 3501/3501 [07:14<00:00,  8.05it/s]


epoch 19: train_loss=0.003606 | test_loss=0.003725
✔ improved test loss; saved best model to ./data/diffusion_params_v1_deep/best_model.pth
saved image at ./data/diffusion_params_v1_deep/image_ep19_w0.0.png
saved gif at ./data/diffusion_params_v1_deep/gif_ep19_w0.0.gif
saved image at ./data/diffusion_params_v1_deep/image_ep19_w0.5.png
saved gif at ./data/diffusion_params_v1_deep/gif_ep19_w0.5.gif
saved image at ./data/diffusion_params_v1_deep/image_ep19_w2.0.png
saved gif at ./data/diffusion_params_v1_deep/gif_ep19_w2.0.gif
updated loss curve at ./data/diffusion_params_v1_deep/loss_curve.png
saved final model at ./data/diffusion_params_v1_deep/model_final.pth


In [33]:
import torch
import numpy as np
import matplotlib.pyplot as plt

def _to_top_pair(cond_or_top):
    """
    Accepts:
      - cond_or_top [B,4,2]: (modes, weights) → returns top pair [B,2] via [:,0,:]
      - cond_or_top [B,2]: already top pair
    """
    if cond_or_top.dim() == 3:
        assert cond_or_top.size(2) == 2 and cond_or_top.size(1) >= 1, \
            f"Expected [B,4,2]-like, got {tuple(cond_or_top.shape)}"
        return cond_or_top[:, 0, :]  # [B,2]
    assert cond_or_top.dim() == 2 and cond_or_top.size(1) == 2, \
        f"Expected [B,2], got {tuple(cond_or_top.shape)}"
    return cond_or_top

@torch.no_grad()
def compare_waveguides_3gens(
    ddpm, test_loader, device, guide_w=2.0, n_rows=12,
    save_path=None, binarize=False, thresh=0.5
):
    """
    For each of the first n_rows items from the test loader, generate 3 waveguides
    conditioned on the same (top_pair, params), and display as:
        Real (red) | Gen 1 (black) | Gen 2 (black) | Gen 3 (black)

    Expects test_loader to yield either:
      (cond:[B,4,2], params:[B,4], x_real:[B,1,32,32])  OR
      (top_pair:[B,2], params:[B,4], x_real:[B,1,32,32])
    """
    ddpm.eval()

    # Grab one test batch
    try:
        cond_or_top, params, x_real = next(iter(test_loader))
    except StopIteration:
        raise RuntimeError("test_loader is empty.")

    n = min(n_rows, cond_or_top.shape[0])
    cond_or_top = cond_or_top[:n].to(device)
    params      = params[:n].to(device)
    x_real      = x_real[:n].to(device)

    # Convert to [n,2] top pair
    top_pair = _to_top_pair(cond_or_top)  # [n,2]

    # Repeat each condition 3× to get 3 generated samples per item
    num_gens     = 3
    top_pair_rep = top_pair.repeat_interleave(num_gens, dim=0)   # [n*3,2]
    params_rep   = params.repeat_interleave(num_gens, dim=0)     # [n*3,4]

    # Sample all at once (DDPM randomness gives different outputs per repeat)
    x_gen_all, _ = ddpm.sample(
        n_sample=n * num_gens,
        size=(1, 32, 32),
        device=device,
        top_pair=top_pair_rep,   # <-- updated
        params=params_rep,
        guide_w=guide_w,
    )  # [n*3,1,32,32]

    # Reshape to [n, 3, 1, 32, 32]
    x_gen_all = x_gen_all.view(n, num_gens, 1, 32, 32)

    # To CPU numpy
    real_np = x_real.detach().cpu().numpy()           # [n,1,32,32]
    gen_np  = x_gen_all.detach().cpu().numpy()        # [n,3,1,32,32]

    # Clamp and optional binarization
    real_np = np.clip(real_np, 0.0, 1.0)
    gen_np  = np.clip(gen_np,  0.0, 1.0)
    if binarize:
        real_np = (real_np >= thresh).astype(np.float32)
        gen_np  = (gen_np  >= thresh).astype(np.float32)

    # Plot: 4 columns => Real + 3 gens
    cols = 4
    fig_h = max(2, n * 1.1)
    fig, axs = plt.subplots(n, cols, figsize=(cols * 2.2, fig_h), squeeze=False)

    for i in range(n):
        # Real (red)
        ax = axs[i, 0]
        ax.imshow(real_np[i, 0], cmap="Reds", vmin=0, vmax=1)
        if i == 0: ax.set_title("Real", fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])

        # Gen 1..3 (black). Use 1 - gen for black-on-white with 'binary' cmap
        for j in range(num_gens):
            ax = axs[i, j + 1]
            ax.imshow(1.0 - gen_np[i, j, 0], cmap="binary_r", vmin=0, vmax=1)
            if i == 0: ax.set_title(f"Gen {j+1}", fontsize=10)
            ax.set_xticks([]); ax.set_yticks([])

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
        plt.close(fig)
        print(f"Saved comparison grid to {save_path}")
    else:
        plt.show()


h5_path='train_test_split.h5'
n_epoch=20
batch_size=256
n_T=400
n_feat=128
lrate=1e-4
drop_prob=0.1
betas=(1e-4, 0.02)
ws_test=(0.0, 0.5, 2.0)
save_model=True
test_eval_fraction=0.25   # portion of test loader to evaluate each epoch

device = "cuda:0" if torch.cuda.is_available() else "cpu"

# ---- Data ----
train_ds = WaveguideDatasetPaired(h5_path, split="train", stats_path=None)
test_ds  = WaveguideDatasetPaired(h5_path, split="test",  stats_path=None)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)

# ---- Model ----
unet = ContextUnet(in_channels=1, n_feat=n_feat, use_time_embed=True)
ddpm = DDPM(nn_model=unet, betas=betas, n_T=n_T, device=device, drop_prob=drop_prob).to(device)
ddpm.load_state_dict(torch.load('/data/diffusion_params_v3_topmode/model_final.pth', map_location=torch.device(device)))

compare_waveguides_3gens(ddpm, test_loader, device, guide_w=2.0, n_rows=12,
                          save_path="/data/diffusion_params_v3_topmode//compare_3gens.png",
                          binarize=True, thresh=0.5)


Saved comparison grid to data/diffusion_params_v1_deep/compare_3gens.png
